In [1]:
# Cell 1: Environment steup, loading data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import torch
import torch.nn as nn


In [2]:
# Cell 2: LSTM Sequence generation (sliding windows)
def create_sequences(df_ticker, seq_length):
    features = ['Close', 'Volume', 'Sentiment', 'SMA_20', 'RSI_14']

    target = 'Close'

    X, y, dates = [], [], []

    for i in range(len(df_ticker) - seq_length):
        seq_x = df_ticker[features].iloc[i : i + seq_length].values 
        seq_y = df_ticker[target].iloc[i + seq_length]
        seq_date = df_ticker['Trading_Date'].iloc[i + seq_length]
        

        X.append(seq_x)
        y.append(seq_y)
        dates.append(seq_date)

    return np.array(X), np.array(y), np.array(dates)


In [3]:
# Cell 3: Building the deep learning model (LSTM)

class StockPredictorLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size=1, dropout_rate=0.2):
        super(StockPredictorLSTM, self).__init__()

        # Core LSTM Layer
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout_rate if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout_rate)

        self.fc1 = nn.Linear(hidden_size, 32)
        self.relu = nn.ReLU()

        # Output prediction layer
        self.fc2 = nn.Linear(32, output_size)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_timestep_out = lstm_out[:, -1, :]

        out = self.dropout(last_timestep_out)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)

        return out


# Init model
INPUT_SIZE = 5
HIDDEN_SIZE = 256  
NUM_LAYERS = 1     
OUTPUT_SIZE = 1   
model = StockPredictorLSTM(
    input_size=INPUT_SIZE, 
    hidden_size=HIDDEN_SIZE, 
    num_layers=NUM_LAYERS, 
    output_size=OUTPUT_SIZE
)
print(model)



StockPredictorLSTM(
  (lstm): LSTM(5, 256, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc1): Linear(in_features=256, out_features=32, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=32, out_features=1, bias=True)
)


In [4]:
# Cell 4: Standardized Triple Loop (V1 LSTM)
import os
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, r2_score
from torch.utils.data import TensorDataset, DataLoader

# --- CONFIGS ---
TICKERS = ['MSFT', 'DIS', 'WMT'] 
SEQ_LENGTHS = [5, 10, 20] 
NUM_RUNS = 10
EPOCHS = 100
BATCH_SIZE = 128
LEARNING_RATE = 0.001

BASE_MODEL_DIR = "models/v1"
BASE_RESULTS_DIR = "results/trainV1"
os.makedirs(BASE_RESULTS_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for seq_len in SEQ_LENGTHS:
    print(f"\n" + "#"*60 + f"\nSTARTING V1 EXPERIMENTS | SEQUENCE LENGTH: {seq_len}\n" + "#"*60)
    SEQ_RESULTS = []
    
    for ticker in TICKERS:
        print(f"\n>>> Processing: {ticker} (Seq: {seq_len})")
        
        file_path = f"datasets/{ticker}/{ticker}_DatasetV1.csv"
        if not os.path.exists(file_path):
            print(f"Error: {file_path} not found.")
            continue
            
        ticker_df = pd.read_csv(file_path)
        ticker_df['Trading_Date'] = pd.to_datetime(ticker_df['Trading_Date'])
        
        # Split by Year
        train_raw = ticker_df[ticker_df['Trading_Date'].dt.year <= 2022].copy()
        test_raw = ticker_df[ticker_df['Trading_Date'].dt.year == 2023].copy()
        
        # Scale: Fit on Train Only
        scaler = MinMaxScaler()
        cols = ['Close', 'Volume', 'Sentiment', 'SMA_20', 'RSI_14']
        train_raw[cols] = scaler.fit_transform(train_raw[cols])
        test_raw[cols] = scaler.transform(test_raw[cols])
        
        # Generate Sequences
        X_train, y_train, _ = create_sequences(train_raw, seq_len)
        X_test, y_test, _ = create_sequences(test_raw, seq_len)
        
        X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
        y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
        X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

        for run in range(1, NUM_RUNS + 1):
            # Single-layer LSTM, 256 units (V1 Architecture)
            model = StockPredictorLSTM(input_size=5, hidden_size=256, num_layers=1).to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
            criterion = nn.MSELoss()
            
            train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
            
            model.train()
            for epoch in range(EPOCHS):
                for xb, yb in train_loader:
                    optimizer.zero_grad()
                    criterion(model(xb), yb).backward()
                    optimizer.step()
            
            model.eval()
            with torch.no_grad():
                preds = model(X_test_t).cpu().numpy().flatten()
            
            r2 = r2_score(y_test, preds)
            mae = mean_absolute_error(y_test, preds)
            
            # Store Metrics
            SEQ_RESULTS.append({'Ticker': ticker, 'Run': run, 'Seq': seq_len, 'R2': r2, 'MAE': mae})
            
            # Save weights (.pth)
            save_path = os.path.join(BASE_MODEL_DIR, ticker)
            os.makedirs(save_path, exist_ok=True)
            torch.save(model.state_dict(), f"{save_path}/v1_run_{run}_seq{seq_len}.pth")
            
            print(f"  Run {run:02d}: R2={r2:.4f}")

    # After each sequence length, save a CSV report
    results_df = pd.DataFrame(SEQ_RESULTS)
    csv_path = os.path.join(BASE_RESULTS_DIR, f"v1_stability_results_{seq_len}.csv")
    results_df.to_csv(csv_path, index=False)
    
    print(f"\n[DONE] Seq {seq_len} results saved to: {csv_path}")
    display(results_df.groupby('Ticker')[['R2', 'MAE']].agg(['mean', 'std']))



############################################################
STARTING V1 EXPERIMENTS | SEQUENCE LENGTH: 5
############################################################

>>> Processing: MSFT (Seq: 5)
  Run 01: R2=0.9757
  Run 02: R2=0.9664
  Run 03: R2=0.9589
  Run 04: R2=0.9780
  Run 05: R2=0.9763
  Run 06: R2=0.9612
  Run 07: R2=0.9708
  Run 08: R2=0.9750
  Run 09: R2=0.9360
  Run 10: R2=0.9739

>>> Processing: DIS (Seq: 5)
  Run 01: R2=0.9026
  Run 02: R2=0.9077
  Run 03: R2=0.8978
  Run 04: R2=0.8913
  Run 05: R2=0.8725
  Run 06: R2=0.9344
  Run 07: R2=0.9274
  Run 08: R2=0.9013
  Run 09: R2=0.8235
  Run 10: R2=0.7994

>>> Processing: WMT (Seq: 5)
  Run 01: R2=0.9354
  Run 02: R2=0.9453
  Run 03: R2=0.9407
  Run 04: R2=0.8194
  Run 05: R2=0.9251
  Run 06: R2=0.8865
  Run 07: R2=0.9307
  Run 08: R2=0.9097
  Run 09: R2=0.9249
  Run 10: R2=0.8726

[DONE] Seq 5 results saved to: results/trainV1\v1_stability_results_5.csv


R2                 MAE          
            mean       std      mean       std
Ticker                                        
DIS     0.885795  0.043217  0.018071  0.003790
MSFT    0.967218  0.012775  0.023000  0.004966
WMT     0.909019  0.039254  0.027614  0.007068


############################################################
STARTING V1 EXPERIMENTS | SEQUENCE LENGTH: 10
############################################################

>>> Processing: MSFT (Seq: 10)
  Run 01: R2=0.9359
  Run 02: R2=0.9590
  Run 03: R2=0.9372
  Run 04: R2=0.9722
  Run 05: R2=0.9638
  Run 06: R2=0.9598
  Run 07: R2=0.9667
  Run 08: R2=0.9510
  Run 09: R2=0.9645
  Run 10: R2=0.9742

>>> Processing: DIS (Seq: 10)
  Run 01: R2=0.8917
  Run 02: R2=0.8931
  Run 03: R2=0.8477
  Run 04: R2=0.7950
  Run 05: R2=0.8976
  Run 06: R2=0.9008
  Run 07: R2=0.8803
  Run 08: R2=0.8228
  Run 09: R2=0.8986
  Run 10: R2=0.9098

>>> Processing: WMT (Seq: 10)
  Run 01: R2=0.8648
  Run 02: R2=0.9007
  Run 03: R2=0.9262
  Run 04: R2=0.9245
  Run 05: R2=0.9206
  Run 06: R2=0.8930
  Run 07: R2=0.8159
  Run 08: R2=0.8203
  Run 09: R2=0.8519
  Run 10: R2=0.9300

[DONE] Seq 10 results saved to: results/trainV1\v1_stability_results_10.csv


R2                 MAE          
            mean       std      mean       std
Ticker                                        
DIS     0.873739  0.038623  0.019326  0.003357
MSFT    0.958431  0.013269  0.025538  0.004647
WMT     0.884798  0.043886  0.031507  0.007365


############################################################
STARTING V1 EXPERIMENTS | SEQUENCE LENGTH: 20
############################################################

>>> Processing: MSFT (Seq: 20)
  Run 01: R2=0.9390
  Run 02: R2=0.9691
  Run 03: R2=0.9613
  Run 04: R2=0.9670
  Run 05: R2=0.9767
  Run 06: R2=0.9714
  Run 07: R2=0.9189
  Run 08: R2=0.9763
  Run 09: R2=0.9700
  Run 10: R2=0.9590

>>> Processing: DIS (Seq: 20)
  Run 01: R2=0.8915
  Run 02: R2=0.9147
  Run 03: R2=0.9166
  Run 04: R2=0.9228
  Run 05: R2=0.8976
  Run 06: R2=0.8472
  Run 07: R2=0.9157
  Run 08: R2=0.8873
  Run 09: R2=0.8971
  Run 10: R2=0.8202

>>> Processing: WMT (Seq: 20)
  Run 01: R2=0.7173
  Run 02: R2=0.9035
  Run 03: R2=0.7333
  Run 04: R2=0.4917
  Run 05: R2=0.8530
  Run 06: R2=0.8752
  Run 07: R2=0.5661
  Run 08: R2=0.8070
  Run 09: R2=0.7389
  Run 10: R2=0.9276

[DONE] Seq 20 results saved to: results/trainV1\v1_stability_results_20.csv


R2                 MAE          
            mean       std      mean       std
Ticker                                        
DIS     0.891064  0.033126  0.016872  0.002750
MSFT    0.960860  0.018341  0.022348  0.005600
WMT     0.761352  0.143513  0.043888  0.015077